# 실험 OE: negative 를 "어느 함수도 아님"으로 학습 + softmax 로 거절 (논문에 없는 추가)
설계·예측은 `EXPERIMENT_DESIGN.md` 4.2절 (돌리기 전에 적어 둠).

- 학습 데이터: ours(A50)와 **완전히 같은** `data/train.jsonl` 4,000개 (positive 2,000 + negative 2,000). 3 epoch → 750 step
- 바뀌는 것 하나: negative 의 loss (`--neg-loss oe`)
  - ours: negative 정답 = `<nexa_20>` → 거절을 21번째 함수처럼 외운다 (매핑 가설)
  - OE: negative 에서는 함수 토큰 20개 확률이 **고를수록** loss 가 작다 (Outlier Exposure, Hendrycks 외 2019). `<nexa_20>` 은 안 씀
  - 전체 loss = positive loss + 0.5 × OE loss (`--oe-lambda 0.5`, 원 논문 값)
- 거절: A00 과 같이 **MSP** = 1 − (함수 토큰 20개 중 최대 확률) 이 문턱값 이상이면 거절
- 채점: **우리 test(800, dev)** 와 **공식 210** 두 채점기. 각각 ① greedy 정답 유무 (`src.evaluate score`) ② softmax 확률 저장 (`src.probs`) ③ 정답 + softmax 채점 (`src.softmax_score`)
- 문턱값: dev(우리 test) positive 의 90% 가 거절되지 않게 dev 에서 정하고, 공식 210 에 그대로 쓴다
- 비교: 같은 채점을 기존 ours·공개 모델에도 한다 (점수 `irr` = `<nexa_20>` 확률, `msp` 둘 다)
- MASSIVE·BFCL 은 이 노트북에서 채점하지 않는다 (설계 문서 7절 5단계 규칙)

런타임: **A100 GPU + 고용량 RAM (80GB)**. 🔑 Secrets 에 `HF_TOKEN` (노트북 액세스 켜기)
결과: Drive `MyDrive/octopus_v2/runs/OE/` (모델), `runs/eval/OE/` (우리 test), `runs/eval_official/OE/` (공식 210)
비교 대상 ours·A00 은 노트북 09 결과가 Drive 에 있으면 생성을 건너뛰고 채점만 다시 한다

## 준비 (그냥 실행)

In [ ]:
# 다른 노트북과 같은 버전 (최신 5.x 는 API 가 바뀌어 에러가 난다)
!pip -q install transformers==4.56.2 accelerate
from huggingface_hub import login
from google.colab import userdata, drive

login(userdata.get("HF_TOKEN"))
drive.mount("/content/drive")

import os
REPO = "/content/octopus_v2_reproduce"
if not os.path.exists(REPO):
    !git clone https://github.com/nahcub/octopus_v2_reproduce.git {REPO}
%cd {REPO}
!git pull
!git log --oneline -1

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import transformers; print("transformers", transformers.__version__)  # 4.56.2 여야 함

DRIVE = "/content/drive/MyDrive/octopus_v2"
RUN_DIR = f"{DRIVE}/runs/OE"                  # 이번에 학습할 모델
DEV_EVAL = f"{DRIVE}/runs/eval"               # 우리 test 결과 (05 노트북과 같은 곳)
OFF_EVAL = f"{DRIVE}/runs/eval_official"      # 공식 210 결과 (06 노트북과 같은 곳)
DEV_TEST = "data/test.jsonl"
OFF_TEST = "data/raw/official/official_test.jsonl"

# 평가할 모델: 이름 -> 폴더 또는 HF 이름. 이름은 결과 폴더 이름
MODELS = {
    "OE": RUN_DIR,
    "ours": f"{DRIVE}/runs/full",       # = A50, negative 를 <nexa_20> 으로 학습
    "A00": f"{DRIVE}/runs/A00",         # negative 없이 학습 (노트북 09)
}
# 모델마다 볼 거절 점수. OE·A00 은 <nexa_20> 을 안 배웠으니 msp 만 의미 있다
SCORES = {"OE": ["msp"], "ours": ["irr", "msp"], "A00": ["msp"]}
# GPU 확인: ours·A00 과 같은 조건(fp32 가중치 + AdamW, batch 16)은 A100 80GB 에서만 들어간다.
# 40GB 는 가중치 10 + 기울기 10 + Adam 상태 20 = 약 40GB 라 batch 를 줄여도 안 들어감 (8-bit Adam 등은 조건이 달라져 비교가 깨짐)
import torch
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
EXTRA = ""
assert GPU_GB > 60, f"GPU {GPU_GB:.0f} GiB: A100 80GB 가 필요하다. 런타임 연결 해제 후 A100 + 고용량 RAM 으로 다시 연결"
print(f"GPU {GPU_GB:.0f} GiB OK")

---
## 1. 학습 데이터 확인
**예측**: OE 학습에 positive·negative 가 몇 개씩 들어갈까?
> (여기에 예측)

In [ ]:
import json, collections
rows = [json.loads(l) for l in open("data/train.jsonl", encoding="utf-8")]
print("전체", collections.Counter(r["kind"] for r in rows))
print("OE 학습: 전부 사용 (ours 와 같은 4,000개) | positive 함수별 개수", sorted(set(collections.Counter(r["function"] for r in rows if r["kind"] == "positive").values())))

## 2. 학습 전 점검 → 짧은 시험 → 본 학습
확인할 것: `[데이터] 4000개 (negative 2000) ... negative 학습 방식: oe (λ=0.5)`. 80GB 가 아니면 준비 셀에서 멈춘다

시험 학습 로그에 `pos_loss` 와 `oe_loss` 가 따로 찍힌다. `oe_loss` 는 처음에 약 3.00 (= log 20, 새 토큰들이 아직 구분이 안 돼서 이미 고름).
positive 를 배우면서 함수 토큰이 뚜렷해지면 잠깐 올라갔다가, negative 에서 다시 고르게 되면서 3.0 근처로 돌아오는 게 기대하는 모양

In [ ]:
!python -m src.train --neg-loss oe --check-only {EXTRA}

In [ ]:
!python -m src.train --neg-loss oe --max-steps 30 --output-dir /content/runs/try_OE {EXTRA}

In [ ]:
# 본 학습: 4,000 ÷ 16 = 250 step × 3 = 750 step (ours 와 같음). 시작·끝 시각을 기록해 둔다 (설계 문서 7절 "학습 1회 시간")
import datetime; print("시작", datetime.datetime.now())
!python -m src.train --neg-loss oe --output-dir {RUN_DIR} {EXTRA}
print("끝", datetime.datetime.now())

In [ ]:
# loss 곡선
import matplotlib.pyplot as plt
logs = [l for l in json.load(open(f"{RUN_DIR}/log_history.json")) if "loss" in l]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot([l["step"] for l in logs], [l["pos_loss"] for l in logs]); ax[0].set_title("positive loss"); ax[0].set_yscale("log")
ax[1].plot([l["step"] for l in logs if "oe_loss" in l], [l["oe_loss"] for l in logs if "oe_loss" in l]); ax[1].axhline(2.9957, ls="--", c="gray")
ax[1].set_title("OE loss (점선 = log 20, 완전히 고를 때)")
for a in ax: a.set_xlabel("step")
plt.show()

---
## 3. 우리 test (800, dev) 채점
세션이 끊겼으면 준비 셀 두 개를 실행하고 여기부터. 생성은 이미 있는 줄은 건너뛰고 이어서 한다 (ours·public 은 05 노트북 결과가 있으면 거의 바로 끝남).

**예측**: OE 의 greedy negative 정확도는? MSP AUROC 는 ours(1.000)·A00(0.991)과 비교해 어디쯤일까?
> (여기에 예측)

In [ ]:
# ① greedy 생성 + 정답 유무 채점  ② softmax 확률 저장
for name, path in MODELS.items():
    out = f"{DEV_EVAL}/{name}"
    print(f"===== {name}: {path}")
    !python -m src.evaluate generate --model "{path}" --test-file {DEV_TEST} --out {out}/preds.jsonl
    !python -m src.evaluate score --preds {out}/preds.jsonl --test-file {DEV_TEST}
    !python -m src.probs generate --model "{path}" --test-file {DEV_TEST} --preds {out}/preds.jsonl --out {out}/probs.jsonl

In [ ]:
# ③ 정답 + softmax 채점. 문턱값은 각 모델 자기 dev 에서 정한다 (여기서는 test = dev 라 낙관적인 숫자)
for name in MODELS:
    for sc in SCORES[name]:
        out = f"{DEV_EVAL}/{name}"
        print(f"\n#################### {name} | 점수 {sc}")
        !python -m src.softmax_score --test-file {DEV_TEST} --scored {out}/scored.jsonl --probs {out}/probs.jsonl --dev-probs {out}/probs.jsonl --score {sc}

---
## 4. 공식 210 채점
문턱값은 3절의 dev(우리 test) probs 에서 정한 값을 그대로 쓴다.

**예측**: 공식 210 에서 dev 문턱값을 옮겨 썼을 때 OE 가 거절하는 positive 는 ours(65/200)·A00(42/200)보다 많을까 적을까?
> (여기에 예측)

In [ ]:
!python -m src.official_bench convert
for name, path in MODELS.items():
    out = f"{OFF_EVAL}/{name}"
    print(f"===== {name}: {path}")
    !python -m src.evaluate generate --model "{path}" --test-file {OFF_TEST} --out {out}/preds.jsonl
    !python -m src.evaluate score --preds {out}/preds.jsonl --test-file {OFF_TEST}
    !python -m src.probs generate --model "{path}" --test-file {OFF_TEST} --preds {out}/preds.jsonl --out {out}/probs.jsonl

In [ ]:
for name in MODELS:
    for sc in SCORES[name]:
        print(f"\n#################### {name} | 점수 {sc}")
        !python -m src.softmax_score --test-file {OFF_TEST} --scored {OFF_EVAL}/{name}/scored.jsonl --probs {OFF_EVAL}/{name}/probs.jsonl --dev-probs {DEV_EVAL}/{name}/probs.jsonl --score {sc}

---
## 5. 나란히 보기
- `greedy`: 기본 생성 그대로 채점 (정답 유무만)
- `t 적용`: 거절 점수가 dev 문턱값 이상이면 거절로 바꿔 채점
- `AUROC`: 거절 점수로 positive / negative 를 가르는 능력 (문턱값 무관)

In [ ]:
def load(d, name, sc):
    return json.load(open(f"{d}/{name}/softmax_{sc}_summary.json", encoding="utf-8"))

for title, d in (("우리 test (dev, 문턱값을 같은 셋에서 정함)", DEV_EVAL), ("공식 210 (dev 문턱값 적용)", OFF_EVAL)):
    print(f"\n== {title}")
    print(f"{'모델:점수':<14}{'AUROC':>8}{'greedy 전체':>12}{'pos':>8}{'neg':>8}{'t 적용 전체':>12}{'pos':>8}{'neg':>8}{'t':>10}")
    for name in MODELS:
        for sc in SCORES[name]:
            s = load(d, name, sc); g, ta = s["greedy"], s["threshold_applied"]
            print(f"{name + ':' + sc:<14}{s['auroc']:>8.3f}{g['acc']:>12.1%}{g['acc_positive']:>8.1%}{g['acc_negative']:>8.1%}"
                  f"{ta['acc']:>12.1%}{ta['acc_positive']:>8.1%}{ta['acc_negative']:>8.1%}{s['threshold']:>10.3g}")

In [ ]:
# 짝지은 bootstrap 으로 차이와 95% 구간 (기준: 첫 번째 = ours 의 <nexa_20>)
args = " ".join(f"--model {n}:{sc} {DEV_EVAL}/{n}/probs.jsonl {DEV_EVAL}/{n}/probs.jsonl" for n in ["ours", "A00", "OE"] for sc in SCORES[n])
!python -m src.refusal_metrics --test {DEV_TEST} {args} | tee {DEV_EVAL}/OE_refusal_metrics.txt
args = " ".join(f"--model {n}:{sc} {OFF_EVAL}/{n}/probs.jsonl {DEV_EVAL}/{n}/probs.jsonl" for n in ["ours", "A00", "OE"] for sc in SCORES[n])
!python -m src.refusal_metrics --test {OFF_TEST} --dev-test {DEV_TEST} {args} | tee {OFF_EVAL}/OE_refusal_metrics.txt

---
## 끝나면
결과는 Drive `runs/eval/OE/`, `runs/eval_official/OE/` 의 `softmax_msp_scored.jsonl` (문항별 확률·판정) 과 `softmax_msp_summary.json`, 학습 기록 `runs/OE/log_history.json`.
랩탑에서 Claude 에게 "OE 결과 분석해줘" 라고 하면 Drive 에서 받아 설계 문서 4.2절 예측과 비교한다.